# SQL for Data Platforms
## Part 6: Semi-Structured Data & JSON in SQL

Real pipelines increasingly ingest data that doesn't arrive as flat columns — API responses,
event/clickstream payloads, webhook bodies. Every modern engine can query JSON directly with SQL,
without a separate parsing step. This module uses SQLite's JSON1 extension; **Part 12** maps the
same ideas onto BigQuery `STRUCT`/`ARRAY`, Snowflake `VARIANT`, and friends.

## Setup

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.execute("PRAGMA foreign_keys = ON")
conn.commit()

def sql(query):
    """Execute a SELECT query and return results as a pandas DataFrame."""
    return pd.read_sql_query(query, conn)

def execute(statement):
    """Execute a DDL or DML statement (CREATE, INSERT, UPDATE, DELETE, etc.)."""
    conn.execute(statement)
    conn.commit()

execute("""
CREATE TABLE customers (
    id               INTEGER  PRIMARY KEY,
    name             VARCHAR(100) NOT NULL,
    email            VARCHAR(100) UNIQUE NOT NULL,
    city             VARCHAR(50),
    membership_level VARCHAR(10)  NOT NULL DEFAULT 'basic'
                         CHECK (membership_level IN ('basic', 'premium', 'vip')),
    created_date     DATE NOT NULL DEFAULT (date('now')),
    phone            VARCHAR(20)
)
""")

execute("""
CREATE TABLE products (
    id             INTEGER PRIMARY KEY,
    name           VARCHAR(100) NOT NULL,
    category       VARCHAR(50)  NOT NULL,
    price          DECIMAL(10,2) NOT NULL CHECK (price >= 0),
    stock_quantity INTEGER       NOT NULL DEFAULT 0 CHECK (stock_quantity >= 0)
)
""")

execute("""
CREATE TABLE employees (
    id         INTEGER PRIMARY KEY,
    name       VARCHAR(100) NOT NULL,
    department VARCHAR(50)  NOT NULL,
    hire_date  DATE         NOT NULL,
    salary     DECIMAL(10,2) NOT NULL CHECK (salary > 0)
)
""")

execute("""
CREATE TABLE orders (
    id          INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date  DATE    NOT NULL DEFAULT (date('now')),
    status      VARCHAR(20) NOT NULL DEFAULT 'pending'
                    CHECK (status IN ('pending','shipped','delivered','cancelled')),
    FOREIGN KEY (customer_id) REFERENCES customers(id)
)
""")

execute("""
CREATE TABLE order_items (
    id         INTEGER PRIMARY KEY,
    order_id   INTEGER       NOT NULL,
    product_id INTEGER       NOT NULL,
    quantity   INTEGER       NOT NULL CHECK (quantity > 0),
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price >= 0),
    FOREIGN KEY (order_id)   REFERENCES orders(id),
    FOREIGN KEY (product_id) REFERENCES products(id)
)
""")

conn.executemany("""
INSERT INTO customers (id, name, email, city, membership_level, created_date)
VALUES (?, ?, ?, ?, ?, ?)
""", [
    (1,  'Alice Martin',   'alice.martin@gmail.com',     'Toronto',   'premium', '2022-03-15'),
    (2,  'Bob Chen',       'bob.chen@yahoo.com',         'Vancouver', 'basic',   '2022-05-20'),
    (3,  'Carol White',    'carol.white@gmail.com',       'Montreal',  'vip',     '2022-07-10'),
    (4,  'David Lee',      'david.lee@hotmail.com',       'Calgary',   'basic',   '2022-09-01'),
    (5,  'Emma Davis',     'emma.davis@gmail.com',        'Ottawa',    'premium', '2022-11-15'),
    (6,  'Frank Torres',   'frank.torres@outlook.com',    'Toronto',   'basic',   '2023-01-08'),
    (7,  'Grace Kim',      'grace.kim@gmail.com',         'Vancouver', 'vip',     '2023-02-20'),
    (8,  'Henry Brown',    'henry.brown@yahoo.com',       'Montreal',  'basic',   '2023-03-12'),
    (9,  'Isabel Santos',  'isabel.santos@gmail.com',     'Toronto',   'premium', '2023-04-05'),
    (10, 'James Wilson',   'james.wilson@hotmail.com',    'Calgary',   'basic',   '2023-05-18'),
    (11, 'Karen Taylor',   'karen.taylor@gmail.com',      'Ottawa',    'premium', '2023-06-22'),
    (12, 'Liam Johnson',   'liam.johnson@outlook.com',    'Toronto',   'vip',     '2023-07-30'),
    (13, 'Maria Garcia',   'maria.garcia@gmail.com',      'Montreal',  'basic',   '2023-08-14'),
    (14, 'Nathan Park',    'nathan.park@yahoo.com',       'Vancouver', 'premium', '2023-09-09'),
    (15, 'Olivia Patel',   'olivia.patel@gmail.com',      'Toronto',   'basic',   '2023-10-01'),
    (16, 'Patrick Murphy', 'patrick.murphy@hotmail.com',  'Calgary',   'basic',   '2023-11-11'),
    (17, 'Quinn Adams',    'quinn.adams@gmail.com',       'Ottawa',    'premium', '2023-12-05'),
    (18, 'Rachel Scott',   'rachel.scott@outlook.com',    'Toronto',   'basic',   '2024-01-18'),
    (19, 'Samuel Nguyen',  'samuel.nguyen@gmail.com',     'Vancouver', 'vip',     '2024-02-28'),
    (20, 'Tanya Roberts',  'tanya.roberts@yahoo.com',     'Montreal',  'basic',   '2024-03-15'),
    (21, 'Umar Ali',       'umar.ali@gmail.com',          'Toronto',   'premium', '2024-04-20'),
    (22, 'Vera Kozlov',    'vera.kozlov@hotmail.com',     'Calgary',   'basic',   '2024-05-30'),
    (23, 'Walter Bell',    'walter.bell@gmail.com',       'Ottawa',    'basic',   '2024-07-10'),
    (24, 'Xena Cruz',      'xena.cruz@yahoo.com',         'Vancouver', 'basic',   '2024-09-01'),
    (25, 'Yuki Tanaka',    'yuki.tanaka@gmail.com',       'Montreal',  'basic',   '2024-11-15'),
])

conn.executemany("""
INSERT INTO products (id, name, category, price, stock_quantity)
VALUES (?, ?, ?, ?, ?)
""", [
    (1,  'Wireless Headphones',  'Electronics', 129.99, 45),
    (2,  'Laptop Stand',         'Electronics',  49.99, 80),
    (3,  'USB-C Hub',            'Electronics',  39.99, 120),
    (4,  'Smartwatch',           'Electronics', 299.99, 25),
    (5,  'Running Jacket',       'Clothing',     89.99, 60),
    (6,  'Yoga Pants',           'Clothing',     54.99, 90),
    (7,  'Wool Sweater',         'Clothing',     75.00, 40),
    (8,  'Denim Jeans',          'Clothing',     69.99, 75),
    (9,  'Organic Coffee 1kg',   'Food',         24.99, 200),
    (10, 'Green Tea 100g',       'Food',         14.99, 300),
    (11, 'Protein Bars x12',     'Food',         34.99, 150),
    (12, 'Olive Oil 500ml',      'Food',         19.99, 250),
    (13, 'Python Programming',   'Books',        44.99, 55),
    (14, 'Data Science Handbook','Books',        54.99, 40),
    (15, 'SQL for Beginners',    'Books',        29.99, 70),
    (16, 'Machine Learning A-Z', 'Books',        49.99, 35),
    (17, 'Yoga Mat',             'Sports',       49.99, 65),
    (18, 'Resistance Bands Set', 'Sports',       29.99, 100),
    (19, 'Kettlebell 16kg',      'Sports',       79.99, 30),
    (20, 'Running Shoes',        'Sports',      139.99, 20),
])

conn.executemany("""
INSERT INTO employees (id, name, department, hire_date, salary)
VALUES (?, ?, ?, ?, ?)
""", [
    (1,  'Alice Foster',   'Sales',       '2019-03-15', 68000.00),
    (2,  'Bob Martinez',   'Sales',       '2020-06-01', 72000.00),
    (3,  'Carol Hughes',   'Sales',       '2021-09-10', 72000.00),
    (4,  'David Okafor',   'Sales',       '2022-11-20', 58000.00),
    (5,  'Eva Schneider',  'Engineering', '2019-01-15', 95000.00),
    (6,  'Frank Liu',      'Engineering', '2020-04-20', 105000.00),
    (7,  'Grace Patel',    'Engineering', '2021-07-01', 112000.00),
    (8,  'Hugo Silva',     'Engineering', '2022-02-15', 95000.00),
    (9,  'Iris Nakamura',  'Engineering', '2023-08-01', 85000.00),
    (10, 'Jake Thompson',  'Marketing',   '2019-11-01', 62000.00),
    (11, 'Kim Anderson',   'Marketing',   '2021-04-15', 70000.00),
    (12, 'Leo Ferreira',   'Marketing',   '2023-01-10', 62000.00),
    (13, 'Mia Campbell',   'HR',          '2020-08-20', 55000.00),
    (14, 'Noa Rosenberg',  'HR',          '2021-12-05', 60000.00),
    (15, 'Omar Diallo',    'HR',          '2022-05-18', 55000.00),
])

conn.executemany("""
INSERT INTO orders (id, customer_id, order_date, status)
VALUES (?, ?, ?, ?)
""", [
    (1,  1,  '2023-01-20', 'delivered'), (2,  1,  '2023-04-15', 'delivered'),
    (3,  1,  '2023-09-10', 'shipped'),   (4,  3,  '2023-02-05', 'delivered'),
    (5,  3,  '2023-06-18', 'delivered'), (6,  3,  '2023-11-22', 'pending'),
    (7,  3,  '2024-03-30', 'shipped'),   (8,  7,  '2023-03-12', 'delivered'),
    (9,  7,  '2023-08-25', 'delivered'), (10, 7,  '2024-01-10', 'pending'),
    (11, 12, '2023-02-28', 'delivered'), (12, 12, '2023-07-14', 'delivered'),
    (13, 12, '2023-12-05', 'shipped'),   (14, 12, '2024-05-20', 'pending'),
    (15, 19, '2023-04-22', 'delivered'), (16, 19, '2023-10-08', 'delivered'),
    (17, 19, '2024-03-15', 'shipped'),   (18, 2,  '2023-03-01', 'delivered'),
    (19, 4,  '2023-05-17', 'delivered'), (20, 5,  '2023-07-28', 'delivered'),
    (21, 6,  '2023-01-30', 'cancelled'), (22, 8,  '2023-06-09', 'delivered'),
    (23, 9,  '2023-08-15', 'delivered'), (24, 10, '2023-09-22', 'shipped'),
    (25, 11, '2023-10-30', 'delivered'), (26, 13, '2023-11-08', 'delivered'),
    (27, 14, '2023-12-15', 'pending'),   (28, 15, '2024-01-25', 'delivered'),
    (29, 16, '2024-02-10', 'shipped'),   (30, 17, '2024-02-28', 'delivered'),
    (31, 18, '2024-03-20', 'cancelled'), (32, 20, '2024-04-05', 'delivered'),
    (33, 21, '2024-04-18', 'shipped'),   (34, 22, '2024-05-02', 'pending'),
    (35, 2,  '2024-06-10', 'delivered'), (36, 4,  '2024-07-05', 'delivered'),
    (37, 5,  '2024-08-20', 'pending'),   (38, 6,  '2024-09-15', 'shipped'),
    (39, 8,  '2024-10-01', 'delivered'), (40, 11, '2024-12-15', 'cancelled'),
])

conn.executemany("""
INSERT INTO order_items (id, order_id, product_id, quantity, unit_price)
VALUES (?, ?, ?, ?, ?)
""", [
    (1,1,1,1,129.99),(2,1,13,1,44.99),(3,2,4,1,279.99),(4,2,17,2,49.99),
    (5,3,9,3,24.99),(6,3,15,1,29.99),(7,4,5,1,89.99),(8,4,6,2,54.99),
    (9,5,2,1,49.99),(10,5,10,2,14.99),(11,6,14,1,54.99),(12,6,18,3,29.99),
    (13,7,7,1,70.00),(14,7,16,1,49.99),(15,8,3,2,39.99),(16,8,11,4,34.99),
    (17,9,20,1,139.99),(18,9,12,2,19.99),(19,10,8,1,69.99),(20,10,19,1,79.99),
    (21,11,1,1,119.99),(22,11,9,2,24.99),(23,12,4,1,299.99),(24,12,13,2,44.99),
    (25,13,6,3,54.99),(26,13,15,1,27.99),(27,14,2,2,49.99),(28,14,17,1,49.99),
    (29,15,5,1,89.99),(30,15,10,3,14.99),(31,16,20,1,129.99),(32,16,14,1,54.99),
    (33,17,7,2,75.00),(34,17,11,2,34.99),(35,18,3,1,39.99),(36,18,18,4,29.99),
    (37,19,16,1,49.99),(38,19,12,3,19.99),(39,20,1,1,129.99),(40,20,8,1,69.99),
    (41,21,4,1,299.99),(42,21,9,2,24.99),(43,22,13,2,44.99),(44,22,15,1,29.99),
    (45,23,6,1,54.99),(46,23,19,1,75.99),(47,24,2,1,49.99),(48,24,10,4,13.99),
    (49,25,5,1,85.00),(50,25,17,2,49.99),(51,26,11,3,34.99),(52,26,20,1,139.99),
    (53,27,7,1,75.00),(54,27,14,1,49.99),(55,28,3,2,39.99),(56,28,16,1,49.99),
    (57,29,8,2,69.99),(58,29,12,2,19.99),(59,30,1,1,129.99),(60,30,18,3,28.99),
    (61,31,4,1,299.99),(62,31,9,1,24.99),(63,32,13,1,44.99),(64,32,6,2,54.99),
    (65,33,2,1,49.99),(66,33,15,2,29.99),(67,34,19,1,79.99),(68,34,10,3,14.99),
    (69,35,5,1,89.99),(70,35,16,1,49.99),(71,36,20,1,139.99),(72,36,11,2,34.99),
    (73,37,7,1,75.00),(74,37,17,1,49.99),(75,38,3,3,37.99),(76,38,12,2,19.99),
    (77,39,14,1,54.99),(78,39,18,4,29.99),(79,40,1,1,129.99),(80,40,8,1,69.99),
])
conn.commit()

print("Store dataset ready:")
for table in ["customers", "products", "employees", "orders", "order_items"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<15} {n:>3} rows")

Store dataset ready:
  customers        25 rows
  products         20 rows
  employees        15 rows
  orders           40 rows
  order_items      80 rows


## Section 1 — Storing and validating JSON

JSON is stored as ordinary `TEXT` — the engine parses it at query time. `json_valid()` checks
whether a string is well-formed JSON before you rely on it.

In [2]:
execute("""
CREATE TABLE customer_events (
    id          INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    event_type  VARCHAR(30) NOT NULL,
    payload     TEXT NOT NULL   -- JSON blob: varies by event_type
)
""")
conn.executemany("""
INSERT INTO customer_events (id, customer_id, event_type, payload) VALUES (?, ?, ?, ?)
""", [
    (1, 1, 'page_view',  '{"page": "/products", "device": "mobile", "duration_sec": 42}'),
    (2, 1, 'add_to_cart','{"product_id": 4, "qty": 1, "price": 299.99}'),
    (3, 7, 'page_view',  '{"page": "/checkout", "device": "desktop", "duration_sec": 118}'),
    (4, 7, 'purchase',   '{"order_total": 174.98, "items": [{"product_id": 13, "qty": 1}, {"product_id": 15, "qty": 1}]}'),
    (5, 3, 'page_view',  '{"page": "/", "device": "mobile", "duration_sec": 9}'),
])
conn.commit()

sql("SELECT id, event_type, json_valid(payload) AS is_valid_json FROM customer_events")

,id,event_type,is_valid_json
0,1,page_view,1
1,2,add_to_cart,1
2,3,page_view,1
3,4,purchase,1
4,5,page_view,1


## Section 2 — Extracting fields with `json_extract`

`json_extract(column, '$.path')` pulls a single value out of a JSON document. SQLite also offers
the shorthand `column ->> '$.path'` (same result, more compact).

In [3]:
sql("""
SELECT id, event_type,
       json_extract(payload, '$.device')      AS device,
       json_extract(payload, '$.duration_sec') AS duration_sec
FROM customer_events
WHERE event_type = 'page_view'
""")

,id,event_type,device,duration_sec
0,1,page_view,mobile,42
1,3,page_view,desktop,118
2,5,page_view,mobile,9


In [4]:
# Shorthand arrow operator — identical result
sql("""
SELECT id, payload ->> '$.device' AS device
FROM customer_events
WHERE event_type = 'page_view'
""")

,id,device
0,1,mobile
1,3,desktop
2,5,mobile


## Section 3 — Expanding a JSON array with `json_each`

`json_each()` is a table-valued function: it turns a JSON array into one row per element, so it can
be joined like any other table. This is how you "unnest" the `items` array inside the `purchase`
event above.

In [5]:
sql("""
SELECT ce.id AS event_id, ce.customer_id,
       je.value ->> '$.product_id' AS product_id,
       je.value ->> '$.qty'        AS quantity
FROM customer_events ce, json_each(ce.payload, '$.items') je
WHERE ce.event_type = 'purchase'
""")

,event_id,customer_id,product_id,quantity
0,4,7,13,1
1,4,7,15,1


## Section 4 — Aggregating over extracted fields

Once a field is extracted, it behaves like any other column — `GROUP BY`, `AVG`, `CASE WHEN` all
work on it normally.

In [6]:
sql("""
SELECT json_extract(payload, '$.device') AS device,
       COUNT(*)                          AS views,
       ROUND(AVG(json_extract(payload, '$.duration_sec')), 1) AS avg_duration_sec
FROM customer_events
WHERE event_type = 'page_view'
GROUP BY device
""")

,device,views,avg_duration_sec
0,desktop,1,118.0
1,mobile,2,25.5


## Best Practices — Semi-Structured Data

- Validate (`json_valid`) before you extract, especially for data from an external API you don't
  control — a malformed payload should fail loudly, not silently return `NULL` for every field.
- Extract the fields you query often into real, typed columns (a view, or a dbt staging model —
  Part 10) rather than re-parsing JSON on every query; keep the raw payload around for fields you
  only need occasionally.
- `json_each` is the local-SQLite name for a pattern every cloud platform has under a different
  name (BigQuery `UNNEST`, Snowflake `LATERAL FLATTEN`, Databricks `explode()`) — see Part 12.

## Next

**Part 7 — The Query Cookbook** returns to flat, relational querying with 11 of the most common
day-to-day patterns.